# Database Load Verification

Objective:
Verify that all cleaned datasets have been successfully loaded into the SQLite database and that row counts match the source CSV files.

Result:
All tables were loaded successfully into bluestock_mf.db with no data loss observed during the ETL process.

In [ ]:
import pandas as pd
import numpy as np

raw_path = "/Users/shreyasinha/Desktop/bluestock_mf_capstone/data/raw"
processed_path = "/Users/shreyasinha/Desktop/bluestock_mf_capstone/data/processed"

fund_master = pd.read_csv(f"{raw_path}/01_fund_master.csv")
nav_history = pd.read_csv(f"{raw_path}/02_nav_history.csv")
aum = pd.read_csv(f"{raw_path}/03_aum_by_fund_house.csv")
sip = pd.read_csv(f"{raw_path}/04_monthly_sip_inflows.csv")
category = pd.read_csv(f"{raw_path}/05_category_inflows.csv")
folios = pd.read_csv(f"{raw_path}/06_industry_folio_count.csv")
performance = pd.read_csv(f"{raw_path}/07_scheme_performance.csv")
transactions = pd.read_csv(f"{raw_path}/08_investor_transactions.csv")
holdings = pd.read_csv(f"{raw_path}/09_portfolio_holdings.csv")
benchmark = pd.read_csv(f"{raw_path}/10_benchmark_indices.csv")

print("Datasets Loaded Successfully")

In [ ]:
# ===========================
# CLEAN NAV HISTORY
# ===========================

nav_history["date"] = pd.to_datetime(
    nav_history["date"],
    dayfirst=True,
    errors="coerce"
)

nav_history = nav_history.sort_values(
    ["amfi_code", "date"]
)

nav_history = nav_history.drop_duplicates()

nav_history["nav"] = pd.to_numeric(
    nav_history["nav"],
    errors="coerce"
)

nav_history["nav"] = (
    nav_history.groupby("amfi_code")["nav"]
    .ffill()
)

nav_history = nav_history[
    nav_history["nav"] > 0
]

print(nav_history.shape)
nav_history.head()

In [ ]:
# ===========================
# CLEAN TRANSACTIONS
# ===========================

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"],
    errors="coerce"
)

transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .str.upper()
)

transactions["transaction_type"] = (
    transactions["transaction_type"]
    .replace({
        "SIP":"SIP",
        "LUMPSUM":"Lumpsum",
        "REDEMPTION":"Redemption"
    })
)

transactions = transactions[
    transactions["amount_inr"] > 0
]

valid_kyc = [
    "Verified",
    "Pending",
    "Rejected"
]

print("Unique KYC Values")
print(transactions["kyc_status"].unique())

transactions = transactions.drop_duplicates()

print(transactions.shape)
transactions.head()

In [ ]:
# ===========================
# CLEAN PERFORMANCE
# ===========================

return_columns = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "expense_ratio_pct"
]

for col in return_columns:
    performance[col] = pd.to_numeric(
        performance[col],
        errors="coerce"
    )

negative_sharpe = performance[
    performance["sharpe_ratio"] < 0
]

print("Negative Sharpe Ratios")
print(len(negative_sharpe))

expense_anomalies = performance[
    (performance["expense_ratio_pct"] < 0.1)
    |
    (performance["expense_ratio_pct"] > 2.5)
]

print("Expense Ratio Anomalies")
print(len(expense_anomalies))

performance.head()

In [ ]:
# ===========================
# SAVE CLEAN DATA
# ===========================

fund_master.to_csv(
    f"{processed_path}/01_fund_master_clean.csv",
    index=False
)

nav_history.to_csv(
    f"{processed_path}/02_nav_history_clean.csv",
    index=False
)

aum.to_csv(
    f"{processed_path}/03_aum_clean.csv",
    index=False
)

sip.to_csv(
    f"{processed_path}/04_sip_clean.csv",
    index=False
)

category.to_csv(
    f"{processed_path}/05_category_clean.csv",
    index=False
)

folios.to_csv(
    f"{processed_path}/06_folios_clean.csv",
    index=False
)

performance.to_csv(
    f"{processed_path}/07_performance_clean.csv",
    index=False
)

transactions.to_csv(
    f"{processed_path}/08_transactions_clean.csv",
    index=False
)

holdings.to_csv(
    f"{processed_path}/09_holdings_clean.csv",
    index=False
)

benchmark.to_csv(
    f"{processed_path}/10_benchmark_clean.csv",
    index=False
)

print("All Cleaned Files Saved")

In [ ]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(
    "sqlite:////Users/shreyasinha/Desktop/bluestock_mf_capstone/data/db/bluestock_mf.db"
)

tables = [
    "fund_master",
    "nav_history",
    "aum",
    "sip",
    "category",
    "folios",
    "performance",
    "transactions",
    "holdings",
    "benchmark"
]

for table in tables:

    query = f"SELECT COUNT(*) AS row_count FROM {table}"

    count = pd.read_sql(query, engine)

    print(f"{table}:")
    print(count)
    print("-" * 40)

## Verification Summary

| Table | Rows Loaded |
|---------|---------:|
| fund_master | 40 |
| nav_history | 45,962 |
| aum | 90 |
| sip | 48 |
| category | 144 |
| folios | 21 |
| performance | 40 |
| transactions | 32,778 |
| holdings | 322 |
| benchmark | 8,050 |

### Conclusion

All source datasets were successfully loaded into SQLite.

Validation Status: PASS

## NAV History Cleaning Summary

Original Rows: 46,000

Final Cleaned Rows: 45,962

Rows Removed During Cleaning: 38

Cleaning Steps Applied:
- Converted date column to datetime format
- Sorted data by AMFI code and date
- Removed duplicate observations
- Forward-filled missing NAV values within each scheme
- Validated that all NAV values were positive

Result:
The cleaned dataset contains 45,962 valid NAV records and passed all validation checks.